In [7]:
import sys

from brian2 import Hz
import numpy as np 
import pandas as pd 
import os 
import pickle

from brian2 import Hz,mV
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[4]
sys.path.insert(0, str(REPO_ROOT / "code"))
from model_unified_ver import run_exp
from model_unified_ver import default_params as params
import utils as utl
from fitting import hill
from analysis_of_simulation_result import *

from make_network import *
av1a1 = [720575940623041549,720575940622894616,720575940626958878,720575940633984924,720575940611137742,720575940627192337]


labial_cluster = pd.read_parquet(REPO_ROOT/'data/labial_cluster_info_v783.parquet')


In [2]:
# av1a1 
cell_in_network_per_type = {}
rank_df = pd.read_parquet(REPO_ROOT/'figure5/figure5Q,R/data/labial_geosmin_rank.parquet')
in_network_av1a1_labial = rank_df[np.any(rank_df<=20,axis=1)].index.values.astype(int)
in_network_av1a1_labial_cell = target_ids_valid_all[np.isin(g_info_all,in_network_av1a1_labial)]

# av1a1 
rank_df = pd.read_parquet(REPO_ROOT/'figure5/figure5Q,R/data/tarsal_sweet_rank.parquet')
in_network_av1a1_tarsal = rank_df[np.any(rank_df<=20,axis=1)].index.values.astype(int)
in_network_av1a1_tarsal_cell = target_ids_valid_all[np.isin(g_info_all,in_network_av1a1_tarsal)]


inhibitory_neurons1 = [c for c in in_network_av1a1_labial_cell if np.sum(syn_df[syn_df.Presynaptic_ID==c]['Excitatory']<0)>0]#cell_in_network_per_type['av1a1']
inhibitory_type_labial = list(np.unique([fid2g[x] for x in inhibitory_neurons1]).astype(int))
inhibitory_neurons2 = [c for c in in_network_av1a1_tarsal_cell if np.sum(syn_df[syn_df.Presynaptic_ID==c]['Excitatory']<0)>0]#cell_in_network_per_type['av1a1']
inhibitory_type_tarsal = list(np.unique([fid2g[x] for x in inhibitory_neurons2]).astype(int))


inhibitory_type = np.union1d(inhibitory_type_labial,inhibitory_type_tarsal)
inhibitory = inhibitory_type
inhibitory = {}
inhibitory['atGRN+TPN1'] = inhibitory_neurons2
inhibitory['L1+L2+L3'] = inhibitory_neurons1

In [8]:
params_opt_labial = pickle.load(open(REPO_ROOT/'figure4/figure4F-K.MN9_PER_fitting/fitting_result/labial/result_av1a1_170.pkl','rb'))

V_l1l2,K_l1l2,n_l1l2,V_l3,K_l3,n_l3,k_act_L,r0_L,h_L = params_opt_labial.x


firing = [(0,0),*[(np.round(hill(c,V_l1l2,K_l1l2,n_l1l2),1),np.round(hill(c,V_l3,K_l3,n_l3),1)) for c in [10,50,100,500]]]

print(firing)

[(0, 0), (9.1, 4.3), (19.9, 15.7), (25.7, 23.5), (37.9, 39.2)]


In [4]:
interest_neurons = np.union1d(inhibitory['L1+L2+L3'],inhibitory['atGRN+TPN1'])
interest_neurons_2_id = {i:j for j,i in enumerate(interest_neurons)}


In [5]:
pwd

'/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation'

In [8]:
for ff1,ff2 in firing[1:]:
    for p in np.arange(0,1,0.05):
        percent = f'{np.round(p*100,2)}%'
        params['w_syn'] = 0.275*mV
        params['n_run'] = 100
        config = {
            'path_res'  : f'{os.getcwd()}/result/{percent}',                              # directory to store results
            'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',    
            'path_con'  : f'{os.getcwd()}/data/synapse_edges_rm_labial_{percent}.parquet',    # connectivity data
            'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
        }
        if percent not in os.listdir(f'{os.getcwd()}/result'):
            os.mkdir(config['path_res'])
        if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result/{percent}'):
            os.mkdir(f"{config['path_res']}/input_spike_pattern")
        if 'params' not in os.listdir(f'{config["path_res"]}'):
            os.mkdir(f'{config["path_res"]}/params')

        pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


        neu_exc = labial_c2g['L1']+labial_c2g['L2']

        neu_add = [labial_c2g['L3'],interest_neurons]
        except_ids = [interest_neurons]
        except_wsyn = [0*mV]
        neu_slnc = []

        spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_L1L2&L3/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
        predetermined_input_or_not = [1,1,0]
        params['r_poi1'] = ff1 * Hz
        params['r_poi2'] = ff2 * Hz
        params['r_poi3'] = 0 * Hz
        run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_0Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_input_Ids=except_ids,Except_input_w_syn=except_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)

>>> Experiment:     9.1Hz_4.3Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/0.0%/9.1Hz_4.3Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   63 s
>>> Experiment:     9.1Hz_4.3Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/5.0%/9.1Hz_4.3Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   76 s
>>> Experiment:     9.1Hz_4.3Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/10.0%/9.1Hz_4.3Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   66 s
>>> Experiment:     9.1Hz_4.3Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptib

    Elapsed time:   99 s
>>> Experiment:     19.9Hz_15.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/45.0%/19.9Hz_15.7Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   98 s
>>> Experiment:     19.9Hz_15.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/50.0%/19.9Hz_15.7Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     19.9Hz_15.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/55.0%/19.9Hz_15.7Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     19.9Hz_15.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin

    Elapsed time:   97 s
>>> Experiment:     25.7Hz_23.5Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/90.0%/25.7Hz_23.5Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     25.7Hz_23.5Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/95.0%/25.7Hz_23.5Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     37.9Hz_39.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/0.0%/37.9Hz_39.2Hz_0Hz.parquet
    Excited neurons: 96
    Elapsed time:   95 s
>>> Experiment:     37.9Hz_39.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_

In [9]:
for ff1,ff2 in firing[1:]:
    for p in np.arange(0,1,0.05):
        percent = f'{np.round(p*100,2)}%'
        params['w_syn'] = 0.275*mV
        params['n_run'] = 100
        config = {
            'path_res'  : f'{os.getcwd()}/result/{percent}',                              # directory to store results
            'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',    
            'path_con'  : f'{os.getcwd()}/data/synapse_edges_rm_labial_{percent}.parquet',    # connectivity data
            'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
        }
        if percent not in os.listdir(f'{os.getcwd()}/result'):
            os.mkdir(config['path_res'])
        if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result/{percent}'):
            os.mkdir(f"{config['path_res']}/input_spike_pattern")
        if 'params' not in os.listdir(f'{config["path_res"]}'):
            os.mkdir(f'{config["path_res"]}/params')

        pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


        neu_exc = labial_c2g['L1']+labial_c2g['L2']

        neu_add = [labial_c2g['L3'],interest_neurons]
        except_ids = [interest_neurons]
        except_wsyn = [0*mV]
        neu_slnc = []
        
        # stimulate sweet neurons at the predetermined freq. at each concentration.
        spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_L1L2&L3/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'

        d = pickle.load(open(spike_path,'rb'))
        
        # stimulate inhibitory neurons at predetermined firing rate (Geosmin only condition). 
        av1a1_spike_path = f'../../activation_of_same_set_of_interneurons/inhibitory_spike_data/{0}Hz_{0}Hz_170Hz.pkl'
        df_av1a1 = pickle.load(open(av1a1_spike_path,'rb'))

        spike_df = [{1:d[i][1],2:d[i][2],3:df_av1a1[i][3]} for i in range(params['n_run'])]

        with open(f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl','wb') as f:
            pickle.dump(spike_df,f)

        spike_path = f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
        predetermined_input_or_not = [1,1,1]
        params['r_poi1'] = ff1 * Hz
        params['r_poi2'] = ff2 * Hz
        params['r_poi3'] = 170 * Hz
        run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_170Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_input_Ids=except_ids,Except_input_w_syn=except_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)

>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/0.0%/9.1Hz_4.3Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   91 s
>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/5.0%/9.1Hz_4.3Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   86 s
>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/10.0%/9.1Hz_4.3Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   53 s
>>> Experiment:     9.1Hz_4.3Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/fig

    Elapsed time:   97 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/45.0%/19.9Hz_15.7Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/50.0%/19.9Hz_15.7Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   98 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/55.0%/19.9Hz_15.7Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   97 s
>>> Experiment:     19.9Hz_15.7Hz_170Hz
    Output file:    /volume_4/research/seongbong/f

    Elapsed time:   97 s
>>> Experiment:     25.7Hz_23.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/85.0%/25.7Hz_23.5Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   95 s
>>> Experiment:     25.7Hz_23.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/90.0%/25.7Hz_23.5Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   96 s
>>> Experiment:     25.7Hz_23.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference_real_final/3. Redundancy/labial/simulation/result/95.0%/25.7Hz_23.5Hz_170Hz.parquet
    Excited neurons: 96
    Elapsed time:   96 s
>>> Experiment:     37.9Hz_39.2Hz_170Hz
    Output file:    /volume_4/research/seongbong/f